In [49]:
import xml.etree.ElementTree as ET
import json
import os
from collections import defaultdict
import csv

In [50]:
def parseXML(xmlfilepath, outputjsonpath):
    print(xmlfilepath) 
  
    # create element tree object 
    try:
        tree = ET.parse(xmlfilepath)
    except ET.ParseError:
        return "Cannot process due to bad format, skipped", {}

  
    # get root element 
    root = tree.getroot() 
  
    # create empty dict for data
    extracted_data = defaultdict()
    extracted_data['ucid'] = ""
    extracted_data['country'] = ""
    extracted_data['doc-number'] = ""
    extracted_data['kind'] = ""
    extracted_data['lang'] = ""
    extracted_data['family-id'] = ""
    extracted_data['status'] = ""
    extracted_data['date'] = ""
    extracted_data['format'] = ""
    extracted_data['is_representative'] = ""
    extracted_data['intention-to-grant-date'] = ""
    extracted_data['term-of-grant'] = []
    extracted_data['main-classification'] = ""
    extracted_data['further-classification'] = ""
    extracted_data['classification-ipcr'] = []
    extracted_data['invention-title'] = ""
    extracted_data['citations'] = []
    extracted_data['child-doc'] = ""
    extracted_data['parent-doc'] = ""
    extracted_data['applicants'] = []
    extracted_data['inventors'] = []
    extracted_data['agents'] = []
    extracted_data['ep-contracting-states'] = []
    extracted_data['abstract'] = []
    extracted_data['description'] = []
    extracted_data['claims'] = []


    extracted_data['ucid'] = root.attrib['ucid']
    extracted_data['country'] = root.attrib['country']
    extracted_data['doc-number'] = root.attrib['doc-number']
    extracted_data['kind'] = root.attrib['kind']
    extracted_data['lang'] = root.attrib['lang']
    extracted_data['priority-claims'] = []
    if extracted_data['lang'] != "EN":
        return "File not in English, skipped.", {}
    
    if "family-id" in root.attrib:
        extracted_data['family-id'] = root.attrib['family-id']

    if root.attrib['status']:
        extracted_data['status'] = root.attrib['status']
    extracted_data['date'] = root.attrib['date']

    extracted_data['format'] = root.find('./bibliographic-data').find('./application-reference').find('./document-id').attrib['format']
    extracted_data['is_representative'] = root.find('./bibliographic-data').find('./application-reference').attrib['is-representative']

    if root.find('./bibliographic-data').find('./priority-claims') is not None:
        for item in root.find('./bibliographic-data').find('./priority-claims').findall("./priority-claim"):
            if item.find('./document-id').attrib['format'] != 'epo':
                continue
            pri_claim = {}

            pri_claim['ucid'] = ""
            # ucid of priority claim (doesn't always exist)
            if 'ucid' in item.attrib:
                ucid = item.attrib['ucid']
                pri_claim['ucid'] = ucid

            pri_claim['doc-number'] = item.find('./document-id').find('./doc-number').text
            pri_claim['date'] = item.find('./document-id').find('./date').text
            if item.find('./document-id').attrib['status'] is not None:
                pri_claim['status'] = item.find('./document-id').attrib['status']
            pri_claim['country'] = item.find('./document-id').find('./country').text
            extracted_data['priority-claims'].append(pri_claim)

    if root.find('./bibliographic-data').find('./dates-of-public-availability') is not None:
        if root.find('./bibliographic-data').find('./dates-of-public-availability').find('./intention-to-grant-date') is not None:
            extracted_data['intention-to-grant-date'] = root.find('./bibliographic-data').find('./dates-of-public-availability').find('./intention-to-grant-date').find('./date').text
    
    if root.find('./bibliographic-data').find('./term-of-grant') is not None:
        if root.find('./bibliographic-data').find('./term-of-grant').find('./lapse-of-patent') is not None:
            for lapse in root.find('./bibliographic-data').find('./term-of-grant').findall('./lapse-of-patent'):
                lapse_info = {}
                lapse_info['country'] = lapse.find('./document-id').find('./country').text
                lapse_info['date'] = lapse.find('./document-id').find('./date').text
                extracted_data['term-of-grant'].append(lapse_info)
    

    for item in root.find('./bibliographic-data').find('./technical-data'):
        if item.find('./main-classification') is not None:
            extracted_data['main-classification'] = item.find('./main-classification').text
        if item.find('./further-classification') is not None:
            extracted_data['further-classification'] = item.find('./further-classification').text
    
        if item.find('./classification-ipcr') is not None:
            for ipcr in item.findall('./classification-ipcr'):
                extracted_data['classification-ipcr'].append(ipcr.text.strip())
        
        if item.find('./citations') is not None:
            for cit in item.find('./patent-citations').findall('./patcit'):
                extracted_data['citations'].append(cit.attrib['ucid'])
        
        if "lang" in item.attrib:
            if item.attrib['lang'] == 'EN':
                extracted_data['invention-title'] = item.text
    
    if root.find('./bibliographic-data').find('./related-documents') is not None:
        if root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./child-doc') is not None:
            extracted_data['child-doc'] = root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./child-doc').attrib['ucid']
        if root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./parent-doc') is not None:
            extracted_data['parent-doc'] = root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./parent-doc').attrib['ucid']

    if root.find('./bibliographic-data').find('./parties') is not None:
        for item in root.find('./bibliographic-data').find('./parties'):
            if item.find('./applicant') is not None:
                for applicant in item.findall('./applicant'):
                    if applicant.find('./addressbook').find('./name') is not None:
                        extracted_data['applicants'].append(applicant.find('./addressbook').find('./name').text)
                    if applicant.find('./addressbook').find('./last-name') is not None:
                        extracted_data['applicants'].append(applicant.find('./addressbook').find('./last-name').text)
            if item.find('./inventor') is not None:
                for inventor in item.findall('./inventor'):
                    if inventor.find('./addressbook').find('./name') is not None:
                        extracted_data['inventors'].append(inventor.find('./addressbook').find('./name').text)
                    if inventor.find('./addressbook').find('./last-name') is not None:
                        extracted_data['inventors'].append(inventor.find('./addressbook').find('./last-name').text)
            if item.find('./agent') is not None:
                for agent in item.findall('./agent'):
                    if agent.find('./addressbook').find('./name') is not None:
                        extracted_data['agents'].append(agent.find('./addressbook').find('./name').text)
                    if agent.find('./addressbook').find('./last-name') is not None:
                        extracted_data['agents'].append(agent.find('./addressbook').find('./last-name').text)

    if root.find('./bibliographic-data').find('./international-convention-data') is not None:
        if root.find('./bibliographic-data').find('./international-convention-data').find('./designated-states') is not None:
            if root.find('./bibliographic-data').find('./international-convention-data').find('./designated-states').find('./ep-contracting-states') is not None:
                for state in root.find('./bibliographic-data').find('./international-convention-data').find('./designated-states').find('./ep-contracting-states').findall('./country'):
                        extracted_data['ep-contracting-states'].append(state.text.strip())

    if root.find('./abstract') is not None:
        # if root.find('./bibliographic-data').find('./abstract').attrib['lang'] == "EN":
        for abs in root.find('./abstract').findall('./p'):
            try:
                extracted_data['abstract'].append(abs.text.replace("\n", ""))
            except AttributeError:
                continue

    if root.find('./description') is not None:
        for desc in root.find('./description').findall('./p'):
            try:
                extracted_data['description'].append(desc.find('./pre').text.replace("\n", ""))
            except AttributeError:
                if desc.text is not None:
                    extracted_data['description'].append(desc.text.replace("\n", ""))

    # if root.find('./claims') is not None:
    #     try:
    #         for claim in root.find('./claims').find('./claim').findall('./claim-text'):
    #             extracted_data['claims'].append(claim.text.replace("\n", ""))
    #     except AttributeError:
    #         continue

    if root.find('./claims') is not None:
        for claim in root.find('./claims').find('./claim').findall('./claim-text'):
            try:
                extracted_data['claims'].append(claim.text.replace("\n", ""))
            except AttributeError:
                continue

    with open(outputjsonpath, 'w', encoding='utf-8') as json_file:
        json.dump(extracted_data, json_file, ensure_ascii=False, indent=4)
    return (outputjsonpath+" done", extracted_data)
    

# parseXML("/Users/shwethaiyer/Downloads/01_document_collection/EP 3/000000/26/26/26/EP-0262626-B1.xml")
# parseXML("/Users/shwethaiyer/Downloads/01_document_collection/EP 3/000000/16/38/26/EP-0163826-A1.xml")

In [ ]:
def process_patent_files(root_dir, output_dir, csv_output):
    os.makedirs(output_dir, exist_ok=True)

    data = []
    headers = set()

    for root, _, files in sorted(os.walk(root_dir)):
        for file in files:
            if file.endswith('.xml'):
                input_file_path = os.path.join(root, file)

                relative_path = os.path.relpath(input_file_path, root_dir)
                nested_info = relative_path.replace(os.sep, "_").replace(".xml", ".json")
                output_file_path = os.path.join(output_dir, nested_info)

                result, result_json = parseXML(input_file_path, output_file_path)
                if result_json != {}:
                    data.append(result_json)
                    headers.update(result_json.keys())
                print(f"Processed: {input_file_path} -> {result}")

    with open(csv_output, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=list(headers))
        writer.writeheader()
        for datum in data:
            writer.writerow(datum)

    print(f"CSV file created at: {csv_output}")

# root directory containing the XML files
root_dir = "/Users/shwethaiyer/Downloads/01_document_collection/EP/000000/" #TODO: CHANGE THIS TO YOURS

# output directory for JSON files
output_dir = "/Users/shwethaiyer/Downloads/patent_data_rag/" #TODO: CHANGE THIS TO YOURS

csv_output = "/Users/shwethaiyer/Downloads/patent_rag_csv.csv" #TODO: CHANGE THIS TO YOURS

process_patent_files(root_dir, output_dir, csv_output)


In [44]:
def parseXML(xmlfilepath):
    print(xmlfilepath) 
  
    # create element tree object 
    try:
        tree = ET.parse(xmlfilepath)
    except ET.ParseError:
        return "Cannot process due to bad format, skipped", {}

  
    # get root element 
    root = tree.getroot() 
  
    # create empty dict for data
    extracted_data = defaultdict()
    extracted_data['ucid'] = ""
    extracted_data['country'] = ""
    extracted_data['doc-number'] = ""
    extracted_data['kind'] = ""
    extracted_data['lang'] = ""
    extracted_data['family-id'] = ""
    extracted_data['status'] = ""
    extracted_data['date'] = ""
    extracted_data['format'] = ""
    extracted_data['is_representative'] = ""
    extracted_data['intention-to-grant-date'] = ""
    extracted_data['term-of-grant'] = []
    extracted_data['main-classification'] = ""
    extracted_data['further-classification'] = ""
    extracted_data['classification-ipcr'] = []
    extracted_data['invention-title'] = ""
    extracted_data['citations'] = []
    extracted_data['child-doc'] = ""
    extracted_data['parent-doc'] = ""
    extracted_data['applicants'] = []
    extracted_data['inventors'] = []
    extracted_data['agents'] = []
    extracted_data['ep-contracting-states'] = []
    extracted_data['abstract'] = []
    extracted_data['description'] = []
    extracted_data['claims'] = []


    extracted_data['ucid'] = root.attrib['ucid']
    extracted_data['country'] = root.attrib['country']
    extracted_data['doc-number'] = root.attrib['doc-number']
    extracted_data['kind'] = root.attrib['kind']
    extracted_data['lang'] = root.attrib['lang']
    extracted_data['priority-claims'] = []
    if extracted_data['lang'] != "EN":
        return "File not in English, skipped.", {}
    
    if "family-id" in root.attrib:
        extracted_data['family-id'] = root.attrib['family-id']

    if root.attrib['status']:
        extracted_data['status'] = root.attrib['status']
    extracted_data['date'] = root.attrib['date']

    extracted_data['format'] = root.find('./bibliographic-data').find('./application-reference').find('./document-id').attrib['format']
    extracted_data['is_representative'] = root.find('./bibliographic-data').find('./application-reference').attrib['is-representative']

    if root.find('./bibliographic-data').find('./priority-claims') is not None:
        for item in root.find('./bibliographic-data').find('./priority-claims').findall("./priority-claim"):
            if item.find('./document-id').attrib['format'] != 'epo':
                continue
            pri_claim = {}

            pri_claim['ucid'] = ""
            # ucid of priority claim (doesn't always exist)
            if 'ucid' in item.attrib:
                ucid = item.attrib['ucid']
                pri_claim['ucid'] = ucid

            pri_claim['doc-number'] = item.find('./document-id').find('./doc-number').text
            pri_claim['date'] = item.find('./document-id').find('./date').text
            if item.find('./document-id').attrib['status'] is not None:
                pri_claim['status'] = item.find('./document-id').attrib['status']
            pri_claim['country'] = item.find('./document-id').find('./country').text
            extracted_data['priority-claims'].append(pri_claim)

    if root.find('./bibliographic-data').find('./dates-of-public-availability') is not None:
        if root.find('./bibliographic-data').find('./dates-of-public-availability').find('./intention-to-grant-date') is not None:
            extracted_data['intention-to-grant-date'] = root.find('./bibliographic-data').find('./dates-of-public-availability').find('./intention-to-grant-date').find('./date').text
    
    if root.find('./bibliographic-data').find('./term-of-grant') is not None:
        if root.find('./bibliographic-data').find('./term-of-grant').find('./lapse-of-patent') is not None:
            for lapse in root.find('./bibliographic-data').find('./term-of-grant').findall('./lapse-of-patent'):
                lapse_info = {}
                lapse_info['country'] = lapse.find('./document-id').find('./country').text
                lapse_info['date'] = lapse.find('./document-id').find('./date').text
                extracted_data['term-of-grant'].append(lapse_info)
    

    for item in root.find('./bibliographic-data').find('./technical-data'):
        if item.find('./main-classification') is not None:
            extracted_data['main-classification'] = item.find('./main-classification').text
        if item.find('./further-classification') is not None:
            extracted_data['further-classification'] = item.find('./further-classification').text
    
        if item.find('./classification-ipcr') is not None:
            for ipcr in item.findall('./classification-ipcr'):
                extracted_data['classification-ipcr'].append(ipcr.text.strip())
        
        if item.find('./citations') is not None:
            for cit in item.find('./patent-citations').findall('./patcit'):
                extracted_data['citations'].append(cit.attrib['ucid'])
        
        if "lang" in item.attrib:
            if item.attrib['lang'] == 'EN':
                extracted_data['invention-title'] = item.text
    
    if root.find('./bibliographic-data').find('./related-documents') is not None:
        if root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./child-doc') is not None:
            extracted_data['child-doc'] = root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./child-doc').attrib['ucid']
        if root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./parent-doc') is not None:
            extracted_data['parent-doc'] = root.find('./bibliographic-data').find('./related-documents').find('./relation').find('./parent-doc').attrib['ucid']

    if root.find('./bibliographic-data').find('./parties') is not None:
        for item in root.find('./bibliographic-data').find('./parties'):
            if item.find('./applicant') is not None:
                for applicant in item.findall('./applicant'):
                    if applicant.find('./addressbook').find('./name') is not None:
                        extracted_data['applicants'].append(applicant.find('./addressbook').find('./name').text)
                    if applicant.find('./addressbook').find('./last-name') is not None:
                        extracted_data['applicants'].append(applicant.find('./addressbook').find('./last-name').text)
            if item.find('./inventor') is not None:
                for inventor in item.findall('./inventor'):
                    if inventor.find('./addressbook').find('./name') is not None:
                        extracted_data['inventors'].append(inventor.find('./addressbook').find('./name').text)
                    if inventor.find('./addressbook').find('./last-name') is not None:
                        extracted_data['inventors'].append(inventor.find('./addressbook').find('./last-name').text)
            if item.find('./agent') is not None:
                for agent in item.findall('./agent'):
                    if agent.find('./addressbook').find('./name') is not None:
                        extracted_data['agents'].append(agent.find('./addressbook').find('./name').text)
                    if agent.find('./addressbook').find('./last-name') is not None:
                        extracted_data['agents'].append(agent.find('./addressbook').find('./last-name').text)

    if root.find('./bibliographic-data').find('./international-convention-data') is not None:
        if root.find('./bibliographic-data').find('./international-convention-data').find('./designated-states') is not None:
            if root.find('./bibliographic-data').find('./international-convention-data').find('./designated-states').find('./ep-contracting-states') is not None:
                for state in root.find('./bibliographic-data').find('./international-convention-data').find('./designated-states').find('./ep-contracting-states').findall('./country'):
                        extracted_data['ep-contracting-states'].append(state.text.strip())

    if root.find('./abstract') is not None:
        # if root.find('./bibliographic-data').find('./abstract').attrib['lang'] == "EN":
        for abs in root.find('./abstract').findall('./p'):
            extracted_data['abstract'].append(abs.text.replace("\n", ""))

    if root.find('./description') is not None:
        for desc in root.find('./description').findall('./p'):
            try:
                extracted_data['description'].append(desc.find('./pre').text.replace("\n", ""))
            except AttributeError:
                extracted_data['description'].append(desc.text.replace("\n", ""))

    if root.find('./claims') is not None:
        for claim in root.find('./claims').find('./claim').findall('./claim-text'):
            try:
                extracted_data['claims'].append(claim.text.replace("\n", ""))
            except AttributeError:
                continue

    # with open(outputjsonpath, 'w', encoding='utf-8') as json_file:
    #     json.dump(extracted_data, json_file, ensure_ascii=False, indent=4)
    # return (outputjsonpath+" done", extracted_data)
    return extracted_data
    

parseXML("/Users/shwethaiyer/Downloads/01_document_collection/EP/000000/11/13/40/EP-0111340-B1.xml")
# parseXML("/Users/shwethaiyer/Downloads/01_document_collection/EP 3/000000/16/38/26/EP-0163826-A1.xml")

/Users/shwethaiyer/Downloads/01_document_collection/EP/000000/11/13/40/EP-0111340-B1.xml


defaultdict(None,
            {'ucid': 'EP-0111340-B1',
             'country': 'EP',
             'doc-number': '0111340',
             'kind': 'B1',
             'lang': 'EN',
             'family-id': '23782403',
             'status': 'new',
             'date': '19921111',
             'format': 'epo',
             'is_representative': 'NO',
             'intention-to-grant-date': '19910717',
             'term-of-grant': [],
             'main-classification': 'C12Q   1/68',
             'further-classification': '',
             'classification-ipcr': ['C07H  21/04        20060101AFI20051220RMJP',
              'C12Q   1/68        20060101C I20051008RMEP',
              'C12Q   1/68        20060101A I20051008RMEP',
              'G01N  33/50        20060101CLI20051220RMJP',
              'C07H  21/00        20060101A I20051008RMEP',
              'C07H  21/00        20060101C I20051008RMEP',
              'G01N  33/50        20060101ALI20051220RMJP'],
             'invention-tit